In [1]:
import pandas as pd
from pathlib import Path

# Find the data folder whether the kernel runs from the project root or from notebooks/
base = Path.cwd()
data_dir = base / "data" / "raw"
if not data_dir.exists():
    data_dir = base.parent / "data" / "raw"

cols = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR",
        "HS", "AS", "HST", "AST", "HC", "AC", "HF", "AF",
        "HY", "AY", "HR", "AR"]

frames = []
for f in sorted(data_dir.glob("*.csv")):
    df = pd.read_csv(f, encoding="latin-1", usecols=cols)
    df.insert(0, "Season", f.stem)
    frames.append(df)

matches = pd.concat(frames, ignore_index=True)
matches = matches.dropna(subset=["HomeTeam"]).reset_index(drop=True)

print("Shape:", matches.shape)
matches.head()

Shape: (1900, 19)


,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HS,AS,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,2021-22,13/08/2021,Brentford,Arsenal,2,0,H,8,22,3,4,12,8,2,5,0,0,0,0
1,2021-22,14/08/2021,Man United,Leeds,5,1,H,16,10,8,3,11,9,5,4,1,2,0,0
2,2021-22,14/08/2021,Burnley,Brighton,1,2,A,14,14,3,8,10,7,7,6,2,1,0,0
3,2021-22,14/08/2021,Chelsea,Crystal Palace,3,0,H,13,4,6,1,15,11,5,2,0,0,0,0
4,2021-22,14/08/2021,Everton,Southampton,3,1,H,14,6,6,3,13,15,6,8,2,0,0,0


In [2]:
print("Missing values per column:")
print(matches.isna().sum())
print()
print("Duplicate rows:", matches.duplicated().sum())

Missing values per column:
Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HS          0
AS          0
HST         0
AST         0
HF          0
AF          0
HC          0
AC          0
HY          0
AY          0
HR          0
AR          0
dtype: int64

Duplicate rows: 0


In [3]:
matches["Date"] = pd.to_datetime(matches["Date"], dayfirst=True, format="mixed")
matches = matches.sort_values("Date").reset_index(drop=True)

print(matches["Date"].dtype)
print(matches["Date"].min(), "to", matches["Date"].max())

datetime64[us]
2021-08-13 00:00:00 to 2026-05-24 00:00:00


In [4]:
# Total goals and goal difference (home minus away)
matches["TotalGoals"] = matches["FTHG"] + matches["FTAG"]
matches["GoalDiff"] = matches["FTHG"] - matches["FTAG"]

# Readable result column
matches["Result"] = matches["FTR"].map({"H": "Home Win", "D": "Draw", "A": "Away Win"})

# Numeric target for the ML model later: 0 = Home Win, 1 = Draw, 2 = Away Win
matches["Target"] = matches["FTR"].map({"H": 0, "D": 1, "A": 2})

matches[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "Result", "TotalGoals", "GoalDiff", "Target"]].head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,Result,TotalGoals,GoalDiff,Target
0,2021-08-13,Brentford,Arsenal,2,0,Home Win,2,2,0
1,2021-08-14,Man United,Leeds,5,1,Home Win,6,4,0
2,2021-08-14,Burnley,Brighton,1,2,Away Win,3,-1,2
3,2021-08-14,Chelsea,Crystal Palace,3,0,Home Win,3,3,0
4,2021-08-14,Everton,Southampton,3,1,Home Win,4,2,0


In [5]:
print(matches["Result"].value_counts())
print()
print(matches["Result"].value_counts(normalize=True).round(3))

Result
Home Win    839
Away Win    607
Draw        454
Name: count, dtype: int64

Result
Home Win    0.442
Away Win    0.319
Draw        0.239
Name: proportion, dtype: float64


In [6]:
out_dir = data_dir.parent / "processed"
out_dir.mkdir(exist_ok=True)
matches.to_csv(out_dir / "matches_clean.csv", index=False)
print("Saved:", out_dir / "matches_clean.csv")

Saved: /Users/shihan22/football-analysis/data/processed/matches_clean.csv
